In [ ]:
import os, json, time, ast, traceback
from pathlib import Path
import nbformat
import pandas as pd
from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell
from IPython.utils.capture import capture_output

In [ ]:
ROOT = Path.cwd()
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
DEVICE = os.environ.get('NEURAL_DNA_DEVICE', 'auto')
STAGES = ['a_import', 'b_train', 'c_splice', 'd_fit', 'e_infer', 'f_evaluate', 'g_report']
os.environ['NEURAL_DNA_PROFILE'] = PROFILE
os.environ['NEURAL_DNA_DEVICE'] = DEVICE
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
InteractiveShell.instance()
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

In [ ]:
def execute_stage(path):
    # fresh namespace per task; retain rich outputs without starting more kernels
    nb = nbformat.read(path, as_version=4)
    namespace = {'__name__': '__main__', '__file__': str(path)}
    previous = Path.cwd()
    os.chdir(path.parent)
    try:
        for count, cell in enumerate(nb.cells, 1):
            tree = ast.parse(cell.source)
            expression = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
            cell.outputs, cell.execution_count = [], count
            try:
                with capture_output() as captured:
                    exec(compile(tree, str(path), 'exec'), namespace)
                    if expression is not None:
                        value = eval(compile(ast.Expression(expression.value), str(path), 'eval'), namespace)
                        if value is not None:
                            display(value)
                if captured.stdout:
                    cell.outputs.append(nbformat.v4.new_output('stream', name='stdout', text=captured.stdout))
                if captured.stderr:
                    cell.outputs.append(nbformat.v4.new_output('stream', name='stderr', text=captured.stderr))
                for item in captured.outputs:
                    cell.outputs.append(nbformat.v4.new_output('display_data', data=item.data, metadata=item.metadata))
            except Exception as error:
                cell.outputs.append(nbformat.v4.new_output('error', ename=type(error).__name__, evalue=str(error),
                    traceback=traceback.format_exc().splitlines()))
                raise
    finally:
        os.chdir(previous)
        nbformat.write(nb, path)

In [ ]:
# task inputs form one traceable chain
for previous, stage in zip(STAGES, STAGES[1:]):
    path = ROOT / stage / 'input'
    if not path.is_symlink():
        if path.is_dir():
            path.rmdir()
        else:
            path.unlink(missing_ok=True)
        path.symlink_to('../' + previous + '/output', target_is_directory=True)

In [ ]:
timings = []
for stage in STAGES:
    path = ROOT / stage / 'src' / f'{stage}.ipynb'
    start = time.perf_counter()
    execute_stage(path)
    timings.append({'stage': stage, 'seconds': time.perf_counter() - start})
    display(pd.DataFrame(timings))
pd.DataFrame(timings).to_csv(ROOT / f'pipeline_timings_{PROFILE}.csv', index=False)

In [ ]:
pd.read_csv(ROOT / 'g_report' / 'output' / PROFILE / 'summary.csv')[
    ['panel', 'markers', 'steps', 'kind', 'method', 'n', 'block_accuracy', 'boundary_f1', 'perplexity']]